# TO-Agents Lite — run it on a free GPU

A multi-agent topology-optimization pipeline: describe a structural problem in prose,
and a group of agents builds the config, runs a 3D optimization, **looks at the rendered
result with a vision model**, proposes revisions, re-runs, and scores the candidates.

The original needs **four A100s**. This notebook needs none of your own hardware:

| Piece | Where it runs |
|---|---|
| TO solver | this Colab runtime — free T4 if you enable one, else CPU |
| Vision agent | Gemini API |
| AI judge | Gemini API |
| Structured output (prose → JSON) | Gemini API |
| 3D rendering | headless Chromium, CPU |

**Before you start** — `Runtime ▸ Change runtime type ▸ T4 GPU`. It works without one,
but the solver drops to a much smaller mesh.

You need **one** free API key:
- Gemini — https://aistudio.google.com/apikey

(Together is optional — only if you'd rather run the JSON step on Llama-3.3-70B.)

Run the cells in order. Total setup is ~4 minutes.

## 1. What hardware did we get?

In [ ]:
import subprocess

HAS_GPU = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
if HAS_GPU:
    print(subprocess.run(
        'nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv',
        shell=True, capture_output=True, text=True).stdout)
else:
    print('No GPU attached.')
    print('This still works — the solver falls back to pyFANTOM CPU at a smaller mesh.')
    print('For a GPU: Runtime > Change runtime type > T4 GPU, then re-run from here.')

## 2. System dependency

`scikit-sparse` compiles against SuiteSparse, so the headers must exist *before* pip runs.
This is the single most common install failure.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y libsuitesparse-dev > /dev/null
print('SuiteSparse installed')

## 3. Clone and install

`pyFANTOM` is installed straight from its public repo. `cupy-cuda12x` is added only when
a GPU is present — it is deliberately absent from `requirements-lite.txt` so the CPU path
installs cleanly on a machine with no CUDA at all.

In [ ]:
!git clone -q https://github.com/bellastewart/to-agents-lite.git
%cd to-agents-lite

!pip install -q -r requirements-lite.txt

if HAS_GPU:
    # Colab is CUDA 12.x; match the major version.
    !pip install -q cupy-cuda12x
    print('installed cupy for the GPU solver')
else:
    print('skipped cupy — CPU solver')

## 4. Headless browser

The 3D screenshots the vision agent looks at are rendered by k3d into HTML and captured
with headless Chromium. This is pure CPU — it never needed a GPU.

In [ ]:
!playwright install-deps chromium > /dev/null 2>&1
!playwright install chromium 2>&1 | tail -2
print('chromium ready')

## 5. Your API keys

`getpass` keeps these out of the cell output, so saving or sharing this notebook does not
leak them. They live only in this runtime's memory and vanish when it recycles.

**Never** paste a key directly into a cell — that is exactly how keys end up in public git
history.

In [ ]:
import os, getpass

os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key (required): ').strip()

# Optional — press Enter to skip. Only needed if you set TO_TEXT_PROVIDER=together below.
_t = getpass.getpass('Together API key (optional, Enter to skip): ').strip()
if _t:
    os.environ['TOGETHER_API_KEY'] = _t

print('Gemini key set' + (' + Together key set' if _t else ' (Together skipped)'))

## 6. Wire up the models

All three roles run on Gemini, so one key covers everything.

Two things worth knowing:

- Together's **serverless** tier has no vision models. They appear in the catalog but need
  a paid dedicated endpoint, so anything image-related must go to Gemini.
- `instructor`'s `JSON_SCHEMA` mode does **not** work against Google's OpenAI-compat
  endpoint (it rejects `response_format.schema`). `TO_INSTRUCTOR_MODE` is auto-set to
  `TOOLS` for Gemini, which is verified working.

To use Together for the JSON step instead, set `TO_TEXT_PROVIDER='together'` and
`TO_TEXT_MODEL='meta-llama/Llama-3.3-70B-Instruct-Turbo'` (needs the optional key above).

In [ ]:
os.environ.update({
    # 'text' is the STRUCTURED OUTPUT role: prose -> validated JSON, and
    # revisions. It is NOT a chat model — the plain-text `generate` client
    # the agents are constructed with is never actually called.
    'TO_TEXT_PROVIDER':   'gemini',
    'TO_TEXT_MODEL':      'gemini-3.5-flash',
    'TO_VISION_PROVIDER': 'gemini',
    'TO_VISION_MODEL':    'gemini-3.5-flash',
    'TO_JUDGE_PROVIDER':  'gemini',
    'TO_JUDGE_MODEL':     'gemini-3.5-flash',

    'TO_BACKEND':  'auto',   # CUDA when a usable GPU is present, else CPU
    'TO_WEB_PORT': '8080',
    'TO_MAX_RUNS': '3',      # revision rounds; raise once you know the timing
})
print('configured')

## 7. Preflight

`doctor.py` independently checks the solver backend, the renderer, and all three model
roles, then reports which tier you qualify for. **If anything is red, stop here** — its
messages name the fix. Everything downstream assumes this passed.

In [ ]:
!python doctor.py

## 8. Launch

Starts the server and opens it in a Colab window. First run also JIT-compiles the numba
kernels, so give it a moment.

In [ ]:
import time, socket, subprocess

PORT = int(os.environ['TO_WEB_PORT'])

server = subprocess.Popen(
    ['python', 'app.py'], env=os.environ.copy(),
    stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

def up(port, timeout=300):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            return False   # died during startup
        with socket.socket() as s:
            s.settimeout(1)
            if s.connect_ex(('127.0.0.1', port)) == 0:
                return True
        time.sleep(2)
    return False

if up(PORT):
    print(f'server up on :{PORT}')
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)
else:
    print('server failed to start — last 40 log lines:')
    print(open('server.log').read()[-4000:])

## 9. Use it

Pick a built-in example or paste your own description, then press **Run pipeline**.

- **Run** tab — who is speaking and what they are doing, the intended setup diagram,
  the optimization movie, and the final depth/stress renders per revision
- **Activity log** tab — raw stdout
- **Stop** — halts at the current iteration and keeps whatever was produced

### If something goes wrong

```python
print(open('server.log').read()[-5000:])   # server-side traceback
```

### Known limits — read before trusting a result

- **CPU runs are slow and use a reduced mesh** (48×24×24 vs 128×64×64). The agent loop is
  identical; the resolution is not.
- **`LocalFilter` is CUDA-only.** On CPU a per-element `r_min` collapses to its mean and
  prints a warning — length-scale control becomes uniform. It is not the same problem.
- **`MinimumCompliance` drops `E_local`, `local_volume_constraint`, `passive_solid`,
  `passive_void` on CPU**, loudly, because that backend does not accept them.
- **Gemini's free tier has real quotas.** Heavy use returns HTTP 429
  ("exceeded your current quota"). It resets on its own; `doctor.py` reports it
  distinctly from a bad key.
- **Colab recycles idle runtimes.** A long run can be killed mid-way; lower `TO_MAX_RUNS`
  or keep the tab active.

Source, and the measurements behind each of these: https://github.com/bellastewart/to-agents-lite